# 🫁 JORINOVA NEXUS — TB chest X-ray (tb_cxr) detector (fine-tuning)

Detects **pulmonary TB and common CXR findings** (consolidation, cavitation, effusion,
miliary, nodule, fibrosis...). Screening/triage aid — a radiologist reports and
microbiology (sputum smear / GeneXpert / culture) confirms TB.

Produces `tb_cxr.pt`; classes resolve via `backend/ai_services/tb_cxr_findings.json`.
> **Keyless of Kaggle** — data from Roboflow (your free key). **T4 GPU** recommended.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU'

## 2. Install dependencies

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive'); os.makedirs('/content/drive/MyDrive/nexus_tb_cxr', exist_ok=True)
    print('Drive ready')
except Exception as e: print('Drive not mounted:', e)

## 4. Get the training data (Roboflow TB CXR)

In [ ]:
# ── 4. Dataset (Roboflow TB chest X-ray) -> DATA_YAML ──
import os, glob, yaml
from getpass import getpass
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
CANDIDATES = [
    ('capstone-project-jkwwi', 'tb-detection-pigkb'),
    ('test-jaidd', 'tbtest'),
    ('xrdaas', 'chest-xray-yolo')
]
ds = None
for w, p in CANDIDATES:
    try:
        proj = rf.workspace(w).project(p)
        for v in range(1, 12):
            try:
                ds = proj.version(v).download('yolov8', location='/content/data/tb_cxr_rf'); print('OK', w+'/'+p, 'v', v); break
            except Exception: continue
        if ds: break
    except Exception as e:
        print('skip', w+'/'+p, '-', str(e)[:70])
assert ds, 'None downloaded — search https://universe.roboflow.com/search?q=tuberculosis%20chest and add a project.'
DATA_YAML = os.path.join(ds.location, 'data.yaml')
print('DATA_YAML =', DATA_YAML, '| classes:', yaml.safe_load(open(DATA_YAML))['names'])
print('If class names differ from tb_cxr_findings.json keys, tell me and I add aka-aliases.')

## 5. Fine-tune — quality-first (yolov8m @ imgsz 896, 100 epochs)
Bigger backbone + high resolution is the biggest lever for **small CXR findings**
(miliary nodules, thin cavities). CXR-appropriate augmentation: anatomy stays upright and
L/R correct → **no vertical flip, no big rotation**. `batch=8` fits a T4 at 1024; drop to
`batch=4` (or `imgsz=896`) if you hit CUDA OOM.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_tb_cxr/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/detect'
model = YOLO('yolov8m.pt')   # m > s: more capacity for subtle CXR texture (was yolov8s)
# Quality-first: imgsz 896 keeps small nodules/cavities; 100 epochs + cos_lr; AdamW +
# light dropout curbs overfit on a small CXR set. CXR aug: NO flipud, NO big rotation.
model.train(data=DATA_YAML, epochs=100, imgsz=896, batch=8,
            project=PROJECT, name='tb_cxr', pretrained=True, patience=25, plots=True,
            optimizer='AdamW', lr0=0.001, cos_lr=True, dropout=0.15, close_mosaic=20,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, degrees=5, fliplr=0.0, flipud=0.0,
            mosaic=1.0, translate=0.08, scale=0.4)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
# Per-class mAP — a single weak class often drags the overall number down; fix/merge it:
try:
    for i, c in zip(m.box.ap_class_index, m.box.maps): print(f'  {model.names.get(int(i), i)}: mAP50-95={c:.3f}')
except Exception as e:
    print('per-class skipped:', e)

## 6b. RESUME if disconnected

In [ ]:
# 6b. RESUME (run INSTEAD of step 6 if a run disconnects)
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_tb_cxr/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_tb_cxr/**/*.pt', recursive=True) + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)   # restores all args from checkpoint
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_YAML' in globals(), 'Re-run step 4 first so DATA_YAML is set.'
    model = YOLO(ckpt); model.train(data=DATA_YAML, epochs=100, imgsz=896, batch=8,
        project='/content/drive/MyDrive/nexus_tb_cxr/runs', name='tb_cxr_cont', pretrained=True, patience=25,
        optimizer='AdamW', lr0=0.001, cos_lr=True, dropout=0.15, close_mosaic=20,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, degrees=5, fliplr=0.0, flipud=0.0)
m = model.val(); print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export (Drive backup + download)

In [ ]:
import shutil, glob, os
from ultralytics import YOLO
DRIVE = '/content/drive/MyDrive/nexus_tb_cxr'
# IMPORTANT: search ONLY this model's run folder, so a stray /content/runs/detect/train
# (base yolov8 with 80 COCO classes) is never exported by mistake.
cands = sorted(glob.glob(DRIVE + '/runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)
assert cands, 'No tb_cxr run in ' + DRIVE + '/runs/. Re-run the TRAIN cell (step 6); ensure the DATASET cell (step 4) set DATA_YAML.'
best = cands[-1]
names = list(YOLO(best).names.values())
assert not {'person', 'bicycle', 'car'}.issubset(set(names)), \
    ('Picked weights have COCO classes ' + str(names[:4]) + ' -> training did NOT use the tb_cxr dataset. '
     'Re-run step 4 (dataset) then step 6 (training); do NOT export base yolov8 weights.')
print('using weights:', best)
print('classes:', names)
os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/tb_cxr.pt'); print('saved ->', DRIVE + '/tb_cxr.pt')
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename download to **`tb_cxr.pt`** at **`backend/models/tb_cxr/tb_cxr.pt`**, commit, push.
2. Auto-loads for `image_type == "tb_cxr"`; classes resolve via `backend/ai_services/tb_cxr_findings.json`.
3. Needs `INSTALL_ML=1` build to run locally; else Claude vision reads the film.

> ⚠️ Screening/triage only. TB is a clinical + microbiological diagnosis — never rule in/out from the model alone.

## Results — sensitivity & specificity (per class)
Re-validates the trained `best.pt` (no training) and derives sensitivity + specificity from the confusion matrix. Saves `results_metrics.csv` to Drive.

In [ ]:
# NEXUS_RESULT_CELL_V1 — Results: per-class SENSITIVITY & SPECIFICITY (no retraining)
# Run the setup + dataset cell first, then this. It only re-validates best.pt.
import numpy as np, glob, os, csv
from ultralytics import YOLO
DRIVE = '/content/drive/MyDrive/nexus_tb_cxr'
cands = sorted(glob.glob(DRIVE + '/runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)
assert cands, 'No trained best.pt in ' + DRIVE + '/runs/. Run training (or restore from Drive) first.'
model = YOLO(cands[-1]); print('validating:', cands[-1])
m = model.val(data=DATA_YAML)
cm = np.array(m.confusion_matrix.matrix, dtype=float)   # rows = predicted, cols = true
names = list(model.names.values())
total = cm.sum()
rows = []
for i, nm in enumerate(names):
    TP = cm[i, i]
    FP = cm[i, :].sum() - TP        # predicted i, truly not i
    FN = cm[:, i].sum() - TP        # truly i, predicted not i
    TN = total - TP - FP - FN
    sens = TP / (TP + FN) if (TP + FN) else 0.0   # recall / true-positive rate
    spec = TN / (TN + FP) if (TN + FP) else 0.0   # true-negative rate
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    rows.append((nm, sens, spec, prec))
macro = lambda k: sum(r[k] for r in rows) / len(rows)
print('\n%-28s %10s %12s %10s' % ('class', 'sensitivity', 'specificity', 'precision'))
print('-' * 64)
for nm, se, sp, pr in rows:
    print('%-28s %10.3f %12.3f %10.3f' % (nm, se, sp, pr))
print('-' * 64)
print('%-28s %10.3f %12.3f %10.3f' % ('MACRO AVERAGE', macro(1), macro(2), macro(3)))
out = DRIVE + '/results_metrics.csv'
with open(out, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['class', 'sensitivity', 'specificity', 'precision'])
    for r in rows: w.writerow([r[0], round(r[1], 4), round(r[2], 4), round(r[3], 4)])
    w.writerow(['MACRO', round(macro(1), 4), round(macro(2), 4), round(macro(3), 4)])
print('\nsaved ->', out, '(download it and send me the numbers for Table 4.3)')